In [0]:
%pip install --quiet rand-engine faker pandas pyarrow

#### 1. Classes com especificações para geração de dados randômicos (metadata, transformers e debuggers)

Classes especificadas:
- FakeCustomer();
- FakeOrder()

In [0]:
import faker
from rand_engine.core.distinct_core import DistinctCore
from rand_engine.core.numeric_core import NumericCore
from rand_engine.core.datetime_core import DatetimeCore
from rand_engine.core.distinct_utils import DistinctUtils

from datetime import datetime as dt, timedelta

class FakeCustomer:

    def __init__(self):
        self.faker = faker.Faker(locale="pt_BR")

    def metadata(self):
        return {
        "user_id": {
            "method": NumericCore.gen_ints_zfilled,
            "parms": dict(length=14)
        },
        "user_type": {     
            "method": DistinctCore.gen_distincts_untyped,
            "parms": dict(distinct=DistinctUtils.handle_distincts_lvl_1({"standard": 80,"premium":15, "gold": 5, None: 7}, 1))
        },
        "first_name": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[self.faker.first_name() for _ in range(1000)])
        },
        "last_name": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[f"{self.faker.last_name()} {self.faker.last_name()}" for _ in range(10000)])
        },
        "income": {
            "method": NumericCore.gen_floats_normal,
            "parms": dict(mean=10000, std=3000, round=2)
        },
        "balance": {
            "method": NumericCore.gen_floats_normal,
            "parms": dict(mean=5000, std=3000, round=2)
        },
        "profession": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=[self.faker.job() for _ in range(100)])
        },
        "birth_date": dict(
            method=DatetimeCore.gen_datetimes, 
            parms=dict(start='1971-07-05', end='2013-07-06', format_in="%Y-%m-%d", format_out="%d/%m/%Y")
        ),
        "signup_date": dict(
            method=DatetimeCore.gen_timestamps,
            parms=dict(start="01-01-2021", end="31-12-2025", format="%d-%m-%Y")
        )
    }

    def transformer(self, **kwargs):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            df["income"] = np.where(df["income"] < 0, 0, df["income"])
            for k, v in kwargs.items(): df[k] = v
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            return df
        return wrapped_transformer

    def transformer_cdc_update(self, null_rate, **kwargs):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            for col in df.columns:
                df[col] = np.where(np.random.random(df.shape[0]) < null_rate, None, df[col])
            for k, v in kwargs.items(): df[k] = v
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            return df
        return wrapped_transformer
    
    def debugger(self):
        
        
        data = {
            "id": [1, 2, 3],
            "name": ["marco", "gisele", "tauan"],
            "age": [34, 34, 2]
        }
        df = pd.DataFrame(data)
        return df
    

class FakeOrders:

    def __init__(self):
        self.faker = faker.Faker(locale="pt_BR")

    def metadata(self):
        return {
        "order_id": {
            "method": NumericCore.gen_ints_zfilled,
            "parms": dict(length=16)
        },
        "user_id": {
            "method": NumericCore.gen_ints_zfilled,
            "parms": dict(length=10)
        },
        "product_id": {
            "method": NumericCore.gen_ints_zfilled,
            "parms": dict(length=3)
        },
        "user_type": {     
        "method": DistinctCore.gen_distincts_untyped,
        "parms": dict(distinct=DistinctUtils.handle_distincts_lvl_1({"standard": 80,"premium":15, "gold": 5, None: 7}, 1))
        },
        "device": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=["IOS", "Android", "Desktop"])
        },
        "traffic_source": {
            "method": DistinctCore.gen_distincts_typed,
            "parms": dict(distinct=["website", "linkedin", "email"])
        }
        }

    def transformer(self):
        def wrapped_transformer(df: PandasDF) -> PandasDF:
            for col in df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns:
                df[col] = df[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            start_time = dt.now() - timedelta(minutes=10)
            end_time = dt.now()
            random_times = pd.to_datetime(np.random.uniform(start_time.timestamp(), end_time.timestamp(), size=len(df)), unit='s').strftime('%Y-%m-%dT%H:%M:%S')
            df["created_at"] = random_times
            return df
        return wrapped_transformer
    

    def debugger(self):
        data = {
            "id": [1, 2, 3],
            "name": ["marco", "gisele", "tauan"],
            "age": [34, 34, 2]
        }
        df = pd.DataFrame(data)
        return df


### 2. Classe `FilesGenerator`

Exemplos de uso:

#### Gerar um dataframe de amostra
```python
file_generator = FilesGenerator(FakeCustomer())
df_random = file_generator.generate_sample(10)
display(df_random)
```

#### Gera um arquivo em formato JSON no volume mencionado
```python
BASE_PATH, FILE_NAME, EXT = ("/Volumes/prd/demo_volumes/rand_engine_data/logs", "customers", "json")
file_generator = FilesGenerator(FakeCustomer()).config_file_props(BASE_PATH, file_name=FILE_NAME, ext=EXT)
file_generator.write_file()
file_generator.list_files()
```


In [0]:
import time
from datetime import datetime as dt
from random import randint
from uuid import uuid4

import numpy as np
import pandas as pd
from pandas import DataFrame as PandasDF

from rand_engine.main.data_generator import DataGenerator

class FilesGenerator:

  def __init__(self, footprint: FakeCustomer):
    self.footprint = footprint
    self.min_secs, self.max_secs = 10, 20
    

  def setup_output(self, base_path: str, file_name, ext):
    self.base_path = f"{base_path}/{file_name}/{ext}"
    self.file_name = file_name
    self.ext = ext
    return self

  def _get_file_path(self):
    return f"{self.base_path}/{self.file_name}_{str(uuid4())[:8]}.{self.ext}"
  
  def config_period(self, min_secs, max_secs):
    self.min_secs = min_secs
    self.max_secs = max_secs
    return self

  def generate_sample(self, size: int=100):
    return (
      DataGenerator(self.footprint.metadata())
        .generate_pandas_df(size, transformer=self.footprint.transformer())
        .get_df()
    )

  def list_files(self):
    assert self.base_path, "Base path not configured. Run the method setup_output."
    dbutils.fs.mkdirs(self.base_path)
    display(dbutils.fs.ls(self.base_path))

  def delete_files(self):
    assert self.base_path, "Base path not configured. Run the method setup_output."
    dbutils.fs.rm(f"dbfs:{self.base_path}", True)

  def write_debug(self, base_path: str, file_name:str, ext: str="json"):
    assert self.base_path, "Base path not configured. Run the method setup_output."
    df_debug = self.footprint.debugger()
    for i in range(5):
      file_path = f"{self.base_path}_test/{self.file_name}.{self.ext}"
      df_debug.to_json(file_path, orient='records', lines=True)
    print(f"File {file_path} created.")

 
  def write_file(self, size: int=100, const_cols={}):
    file_path = self._get_file_path()
    _ = (
      DataGenerator(self.footprint.metadata()) \
        .generate_pandas_df(size, transformer=self.footprint.transformer(**const_cols))
        .write() \
        .mode("overwrite") \
        .format(f"{self.ext}") \
        .option("compression", None) \
        .load(file_path))
    print(f"File {file_path} created with {size} records.")

  def stream_files(self):
    while 1:
      period = randint(self.min_secs, self.max_secs)
      self.write_file()
      time.sleep(period)


## CDC Generator



In [0]:
from datetime import datetime as dt
from random import randint
from uuid import uuid4
import pytest
import faker
from pandas import DataFrame as PandasDF
import numpy as np
import time
import pandas as pd

from pyspark.sql.functions import lit, coalesce
from rand_engine.main.data_generator import DataGenerator
from rand_engine.core.distinct_core import DistinctCore
from rand_engine.core.numeric_core import NumericCore
from rand_engine.core.datetime_core import DatetimeCore
from rand_engine.core.distinct_utils import DistinctUtils

class CDCGenerator(FilesGenerator):

  def __init__(self, footprint: FakeCustomer, pk_cols=[]):
    self.footprint = footprint
    self.pk_cols = pk_cols
    self.cdc_props = self.default_cdc_properties()

  def default_cdc_properties(self):
    return {
      "INSERT": dict(min_size=100, max_size=200),
      "UPDATE":dict(sample_rate=0.3, null_rate=0.9),
      "DELETE": dict(sample_rate=0.1)
    }

  def set_cdc_properties(self, cdc_properties):
    self.cdc_props = cdc_properties
    return self


  def calculate_rows_to_change(self, sample):
    df = spark.read.format(self.ext).load(self.base_path).filter(coalesce(*self.pk_cols).isNotNull())
    df_ids_inserted = df.select(*self.pk_cols).filter("operation = 'INSERT'").distinct()
    df_ids_deleted = df.select(*self.pk_cols).filter("operation = 'DELETE'").distinct()
    df_pks_to_change = df_ids_inserted.join(df_ids_deleted, on=self.pk_cols, how="leftanti")
    df_pks_to_change = df_pks_to_change.sample(sample).toPandas()
    return df_pks_to_change


  def generate_changes(self, sample, const_cols, null_rate):
    df_pks_to_change = self.calculate_rows_to_change(sample=sample)
    metadata = self.footprint.metadata()
    size = df_pks_to_change.shape[0]
    transformer = self.footprint.transformer_cdc_update(null_rate=null_rate, **const_cols)
    df_data = DataGenerator(metadata).generate_pandas_df(size, transformer).get_df()
    for coluna in self.pk_cols: df_data[coluna] = df_pks_to_change[coluna]
    if null_rate != 1: 
      cols_to_check = [col for col in df_data.columns if col not in self.pk_cols + list(const_cols.keys())]
      mask = ~df_data[cols_to_check].isnull().all(axis=1)
      df_data = df_data[mask]
    return df_data


  def generate_inserts(self):
    operation="INSERT"
    const_cols={"operation": "INSERT", "updated_at": dt.now().strftime("%Y-%m-%dT%H:%M:%S")}
    insert_conf = self.cdc_props["INSERT"]
    file_path = self._get_file_path()
    rand_size = randint(insert_conf["min_size"], insert_conf["max_size"])
    self.write_file(size=rand_size, const_cols=const_cols)


  def generate_cdc(self):
    self.generate_inserts()
    update_conf = self.cdc_props["UPDATE"]
    delete_conf = self.cdc_props["DELETE"]
    update_const_cols ={"operation": "UPDATE", "updated_at": dt.now().strftime("%Y-%m-%dT%H:%M:%S")}
    delete_const_cols ={"operation": "DELETE", "updated_at": dt.now().strftime("%Y-%m-%dT%H:%M:%S")}
    df_update = self.generate_changes(update_conf["sample_rate"], update_const_cols, update_conf["null_rate"])
    df_delete = self.generate_changes(delete_conf["sample_rate"], delete_const_cols, 1)
    df_changes = pd.concat([df_update, df_delete], ignore_index=True)
    file_path = self._get_file_path()
    df_changes.to_json(file_path, orient="records", lines=True)


  def generate_cdc_stream(self, freq=5, rounds=15):
    for i in range(rounds):
      self.generate_cdc()
      time.sleep(freq)